In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

def compute_slo_attainment(json_file, slo_thresholds):
    with open(json_file, "r") as f:
        results = json.load(f)

    submit_times = np.array([r["timestamp_list"][0] for r in results 
                             if r["response"]["meta_data"]["finish_reason"] in ["stop", "length"]])
    finish_times = np.array([r["timestamp_list"][-1] for r in results 
                             if r["response"]["meta_data"]["finish_reason"] in ["stop", "length"]])
    latencies = finish_times - submit_times

    slo_attainment = []
    for thr in slo_thresholds:
        attainment = np.sum(latencies <= thr) / len(latencies)
        slo_attainment.append(attainment * 100)

    return np.array(slo_attainment)

In [ ]:
results_folder = "../results"

result_centralize = f"{results_folder}/centralized_simulation/result.json"
result_decentralize = f"{results_folder}/decentralized_simulation/result.json"
result_single = f"{results_folder}/single_simulation/result.json"

slo_thresholds = [10*i for i in range(10, 50)]

slo_centralize = compute_slo_attainment(result_centralize, slo_thresholds)
slo_decentralize = compute_slo_attainment(result_decentralize, slo_thresholds)
slo_single = compute_slo_attainment(result_single, slo_thresholds)

In [ ]:
plt.figure(figsize=(6,4))

colors = {"single": "#069D12", "decentralize": "#217FE4", "centralize": "#F03F20"}
linestyles = {"centralize": (0, (3, 1, 1, 1)), "decentralize": "-", "single": (0, (5, 1))}

plt.plot(slo_thresholds, slo_centralize, linestyle=linestyles["centralize"], linewidth=2,
             color=colors["centralize"], label="Centralize")
plt.plot(slo_thresholds, slo_decentralize, linestyle=linestyles["decentralize"], linewidth=2,
             color=colors["decentralize"], label="Decentralize")
plt.plot(slo_thresholds, slo_single, linestyle=linestyles["single"], linewidth=2,
             color=colors["single"], label="Single")

plt.xlabel("SLO Threshold (s)", fontsize=14)
plt.ylabel("SLO Attainment (%)", fontsize=14)
plt.ylim(60, 105)
plt.grid(True, linestyle=':', linewidth=1, alpha=0.7)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()